In [1]:
!pip3 install sentence-transformers

In [2]:
!pip3 install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 13.1 MB/s eta 0:00:00


In [3]:
!wget https://s3-geospatial.s3.us-west-2.amazonaws.com/medium_data.csv

--2026-04-30 22:42:46--  https://s3-geospatial.s3.us-west-2.amazonaws.com/medium_data.csv
Resolving s3-geospatial.s3.us-west-2.amazonaws.com (s3-geospatial.s3.us-west-2.amazonaws.com)... 52.92.209.250, 16.12.89.82, 52.92.225.34, ...
Connecting to s3-geospatial.s3.us-west-2.amazonaws.com (s3-geospatial.s3.us-west-2.amazonaws.com)|52.92.209.250|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 599580 (586K) [text/csv]
Saving to: ‘medium_data.csv’

medium_data.csv     100%[===================>] 585.53K  --.-KB/s    in 0.1s    

2026-04-30 22:42:46 (3.95 MB/s) - ‘medium_data.csv’ saved [599580/599580]



In [4]:
!wc -l medium_data.csv

2499 medium_data.csv


In [7]:
import pandas as pd

df = pd.read_csv("medium_data.csv") # excercise whole data set
df.head()

,id,url,title,subtitle,claps,responses,reading_time,publication,date
0,1,https://towardsdatascience.com/not-all-rainbow...,Not All Rainbows and Sunshine: The Darker Side...,Part 1: The Risks and Ethical Issues…,453.0,11,9,Towards Data Science,27-01-2023
1,2,https://towardsdatascience.com/ethics-in-ai-po...,Ethics in AI: Potential Root Causes for Biased...,An alternative approach to understanding bias ...,311.0,3,12,Towards Data Science,27-01-2023
2,3,https://towardsdatascience.com/python-tuple-th...,"Python Tuple, The Whole Truth and Only the Tru...",NaN,188.0,0,24,Towards Data Science,27-01-2023
3,4,https://towardsdatascience.com/dates-and-subqu...,Dates and Subqueries in SQL,Working with dates in SQL,15.0,1,4,Towards Data Science,27-01-2023
4,5,https://towardsdatascience.com/temporal-differ...,Temporal Differences with Python: First Sample...,NaN,10.0,0,13,Towards Data Science,27-01-2023


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2498 entries, 0 to 2497
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            2498 non-null   int64  
 1   url           2498 non-null   object 
 2   title         2498 non-null   object 
 3   subtitle      2073 non-null   object 
 4   claps         2423 non-null   float64
 5   responses     2498 non-null   int64  
 6   reading_time  2498 non-null   int64  
 7   publication   2498 non-null   object 
 8   date          2498 non-null   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 175.8+ KB


In [9]:
df['title'] = df['title'].astype(str).fillna('')
df['subtitle'] = df['subtitle'].astype(str).fillna('')

In [11]:
df['metadata'] = df.apply(lambda row: {'title': row['title'] + " " + row['subtitle']}, axis=1)
df.head()

,id,url,title,subtitle,claps,responses,reading_time,publication,date,metadata
0,1,https://towardsdatascience.com/not-all-rainbow...,Not All Rainbows and Sunshine: The Darker Side...,Part 1: The Risks and Ethical Issues…,453.0,11,9,Towards Data Science,27-01-2023,{'title': 'Not All Rainbows and Sunshine: The ...
1,2,https://towardsdatascience.com/ethics-in-ai-po...,Ethics in AI: Potential Root Causes for Biased...,An alternative approach to understanding bias ...,311.0,3,12,Towards Data Science,27-01-2023,{'title': 'Ethics in AI: Potential Root Causes...
2,3,https://towardsdatascience.com/python-tuple-th...,"Python Tuple, The Whole Truth and Only the Tru...",nan,188.0,0,24,Towards Data Science,27-01-2023,"{'title': 'Python Tuple, The Whole Truth and O..."
3,4,https://towardsdatascience.com/dates-and-subqu...,Dates and Subqueries in SQL,Working with dates in SQL,15.0,1,4,Towards Data Science,27-01-2023,{'title': 'Dates and Subqueries in SQL Working...
4,5,https://towardsdatascience.com/temporal-differ...,Temporal Differences with Python: First Sample...,nan,10.0,0,13,Towards Data Science,27-01-2023,{'title': 'Temporal Differences with Python: F...


In [14]:
from google.colab import userdata

# Use Colab Secrets
API_KEY = userdata.get('pinecone_API')

In [15]:
from pinecone import Pinecone

pc = Pinecone(api_key=API_KEY)

In [16]:
from pinecone import ServerlessSpec

# Spec for free tier plan
spec = ServerlessSpec(
    cloud="aws",
    region="us-east-1"
)

In [17]:
index_name = 'semantic-search-fast'

In [18]:
existing_indexes = [
    index_info["name"] for index_info in pc.list_indexes()
]

existing_indexes

['semantic-search-fast']

In [19]:
import time

# If the following code runs more than once, you might get "The index already existed" error
# In that case, delete the index first: pc.delete_index(index_name)
try:
    pc.delete_index(index_name)
except Exception as e:
    print(e)

pc.create_index(
    index_name,
    dimension=384,  # dimensionality of minilm
    metric='dotproduct',
    spec=spec
)
# wait for index to be initialized
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

In [20]:
from sentence_transformers import SentenceTransformer
import torch

In [21]:
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [22]:
df['values'] = df['metadata'].apply(
    lambda x: model.encode(x['title']).tolist())

In [23]:
df.head()

,id,url,title,subtitle,claps,responses,reading_time,publication,date,metadata,values
0,1,https://towardsdatascience.com/not-all-rainbow...,Not All Rainbows and Sunshine: The Darker Side...,Part 1: The Risks and Ethical Issues…,453.0,11,9,Towards Data Science,27-01-2023,{'title': 'Not All Rainbows and Sunshine: The ...,"[0.004084187094122171, 0.0384955033659935, 0.0..."
1,2,https://towardsdatascience.com/ethics-in-ai-po...,Ethics in AI: Potential Root Causes for Biased...,An alternative approach to understanding bias ...,311.0,3,12,Towards Data Science,27-01-2023,{'title': 'Ethics in AI: Potential Root Causes...,"[-0.07015024125576019, 0.02475566416978836, -0..."
2,3,https://towardsdatascience.com/python-tuple-th...,"Python Tuple, The Whole Truth and Only the Tru...",nan,188.0,0,24,Towards Data Science,27-01-2023,"{'title': 'Python Tuple, The Whole Truth and O...","[-0.05943324789404869, -0.02746076136827469, 0..."
3,4,https://towardsdatascience.com/dates-and-subqu...,Dates and Subqueries in SQL,Working with dates in SQL,15.0,1,4,Towards Data Science,27-01-2023,{'title': 'Dates and Subqueries in SQL Working...,"[0.0023381479550153017, -0.003287975210696459,..."
4,5,https://towardsdatascience.com/temporal-differ...,Temporal Differences with Python: First Sample...,nan,10.0,0,13,Towards Data Science,27-01-2023,{'title': 'Temporal Differences with Python: F...,"[-0.08383706212043762, -0.029564538970589638, ..."


In [24]:
df_upsert = df[['id', 'values', 'metadata']]

In [25]:
df_upsert.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2498 entries, 0 to 2497
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        2498 non-null   int64 
 1   values    2498 non-null   object
 2   metadata  2498 non-null   object
dtypes: int64(1), object(2)
memory usage: 58.7+ KB


In [26]:
df_upsert['id'] = df_upsert['id'].astype(str)

/tmp/ipykernel_23171/2611083708.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_upsert['id'] = df_upsert['id'].astype(str)


In [27]:
df_upsert.head()

,id,values,metadata
0,1,"[0.004084187094122171, 0.0384955033659935, 0.0...",{'title': 'Not All Rainbows and Sunshine: The ...
1,2,"[-0.07015024125576019, 0.02475566416978836, -0...",{'title': 'Ethics in AI: Potential Root Causes...
2,3,"[-0.05943324789404869, -0.02746076136827469, 0...","{'title': 'Python Tuple, The Whole Truth and O..."
3,4,"[0.0023381479550153017, -0.003287975210696459,...",{'title': 'Dates and Subqueries in SQL Working...
4,5,"[-0.08383706212043762, -0.029564538970589638, ...",{'title': 'Temporal Differences with Python: F...


In [28]:
index = pc.Index(index_name)

In [29]:
index.upsert_from_dataframe(df_upsert)

sending upsert requests:   0%|          | 0/2498 [00:00<?, ?it/s]

UpsertResponse(upserted_count=2498, _response_info={'raw_headers': {'date': 'Thu, 30 Apr 2026 22:49:45 GMT', 'content-type': 'application/json', 'content-length': '21', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '5', 'x-pinecone-request-logical-size': '823320', 'x-pinecone-request-latency-ms': '382', 'x-envoy-upstream-service-time': '189', 'x-pinecone-response-duration-ms': '399', 'grpc-status': '0', 'server': 'envoy'}})

In [32]:
xc = index.query(vector=(model.encode("what is ethics in AI")).tolist(),
           top_k=10,
           include_metadata=True,
           include_values=True)

In [33]:
for result in xc['matches']:
    print(result['id'], result['score'], result['metadata'])

1635 0.740670204 {'title': 'Ethics in AI: Potential Root Causes for Biased Algorithms An alternative approach to understanding bias in\xa0data'}
1327 0.71688652 {'title': 'The ethical implications of AI in\xa0design It’s time to talk ethics, chatGPT, Midjourney &\xa0more.'}
2 0.740670204 {'title': 'Ethics in AI: Potential Root Causes for Biased Algorithms An alternative approach to understanding bias in\xa0data'}
662 0.667053223 {'title': 'Ethical Considerations In Machine Learning\xa0Projects Don’t forget these topics when building AI\xa0systems'}
1628 0.651916504 {'title': 'Navigating the Ethical Contours of AI Copy Generators Like\xa0ChatGPT Be prepared to cite the\xa0AI…'}
2030 0.651916504 {'title': 'Navigating the Ethical Contours of AI Copy Generators Like\xa0ChatGPT Be prepared to cite the\xa0AI…'}
1345 0.60717392 {'title': 'Using AI in UX research: a structured and ethical\xa0approach A structured approach for assessing the…'}
370 0.596167564 {'title': 'The Path towards AI Regu